# 🛡️ SURI-CALDERA IDS Practice Lab
## MITRE Caldera + Suricata – Adversary Emulation & Intrusion Detection

---

**Objetivos de la práctica:**
- Emular técnicas del adversario con MITRE Caldera
- Detectar actividad maliciosa en tiempo real con Suricata IDS
- Analizar logs y alertas con Python (pandas, matplotlib)
- Correlacionar ataques con el framework MITRE ATT&CK

**Herramientas:**
| Herramienta | URL | Credenciales |
|-------------|-----|--------------|
| Caldera Web | http://localhost:8888 | admin / admin |
| Jupyter     | http://localhost:8889 | (sin contraseña) |

---

## Sección 1: Verificación del Entorno
Antes de empezar verificamos que todos los servicios estén activos.

In [ ]:
import subprocess
import requests
import json
import os
import time

CALDERA_URL  = 'http://localhost:8888'
CALDERA_KEY  = 'REDADMIN123'
HEADERS      = {'KEY': CALDERA_KEY, 'Content-Type': 'application/json'}
EVE_LOG      = '/var/log/suricata/eve.json'
FAST_LOG     = '/var/log/suricata/fast.log'

def check_service(name):
    result = subprocess.run(['systemctl', 'is-active', name],
                            capture_output=True, text=True)
    status = result.stdout.strip()
    icon = '✅' if status == 'active' else '❌'
    print(f'{icon} {name:12s}: {status}')
    return status == 'active'

def check_url(name, url, key=None):
    try:
        h = {'KEY': key} if key else {}
        r = requests.get(url, headers=h, timeout=5)
        icon = '✅' if r.status_code < 400 else '⚠️'
        print(f'{icon} {name:12s}: HTTP {r.status_code}')
        return r.status_code < 400
    except Exception as e:
        print(f'❌ {name:12s}: {e}')
        return False

print('='*50)
print('  SURI-CALDERA Lab – Environment Check')
print('='*50)
print('\n── Systemd Services ──')
ok_caldera   = check_service('caldera')
ok_suricata  = check_service('suricata')
ok_jupyter   = check_service('jupyter')
print('\n── HTTP Endpoints ──')
ok_api = check_url('Caldera API', f'{CALDERA_URL}/api/v2/health', key=CALDERA_KEY)
print('\n── Log Files ──')
for f in [EVE_LOG, FAST_LOG]:
    exists = os.path.exists(f)
    icon = '✅' if exists else '⚠️'
    print(f'{icon} {f}')

print('\n' + '='*50)
all_ok = ok_caldera and ok_suricata
print('🎯 Lab ready!' if all_ok else '⚠️  Some services need attention – check above')

## Sección 2: Fundamentos de MITRE Caldera

Caldera es una plataforma de emulación de adversarios que implementa el framework MITRE ATT&CK.

**Conceptos clave:**
- **Agent**: proceso que se ejecuta en el objetivo y recibe instrucciones
- **Adversary**: perfil que agrupa técnicas (abilities) en un orden lógico
- **Ability**: técnica atómica mapeada a una táctica/técnica MITRE ATT&CK
- **Operation**: ejecución de un adversario sobre un conjunto de agentes
- **Planner**: algoritmo que decide qué abilities ejecutar y en qué orden


In [ ]:
# ── Listar abilities disponibles ──────────────────────────────────────────────
r = requests.get(f'{CALDERA_URL}/api/v2/abilities', headers=HEADERS, timeout=10)
abilities = r.json() if r.status_code == 200 else []
print(f'Total de abilities disponibles: {len(abilities)}')

# Agrupar por táctica
from collections import Counter
tactics = Counter(a.get('tactic', 'unknown') for a in abilities)
print('\nAbilities por táctica MITRE ATT&CK:')
for tactic, count in sorted(tactics.items(), key=lambda x: -x[1]):
    bar = '█' * min(count, 40)
    print(f'  {tactic:30s} {bar} ({count})')

In [ ]:
# ── Listar adversarios predefinidos ───────────────────────────────────────────
r = requests.get(f'{CALDERA_URL}/api/v2/adversaries', headers=HEADERS, timeout=10)
adversaries = r.json() if r.status_code == 200 else []
print(f'Adversarios disponibles ({len(adversaries)}):\n')
for adv in adversaries[:15]:  # mostrar primeros 15
    name = adv.get('name', 'N/A')
    desc = adv.get('description', '')[:60]
    n_ab = len(adv.get('atomic_ordering', []))
    print(f'  {name:35s} | {n_ab:2d} abilities | {desc}')

## Sección 3: Fundamentos de Suricata

Suricata es un IDS/IPS de alto rendimiento que analiza tráfico de red en tiempo real.

**Archivos de log importantes:**
- `/var/log/suricata/eve.json`: log en formato JSON, contiene todos los eventos
- `/var/log/suricata/fast.log`: log de alertas en formato de texto plano
- `/var/log/suricata/suricata.log`: log del proceso de Suricata


In [ ]:
import subprocess

# ── Versión y configuración de Suricata ───────────────────────────────────────
version = subprocess.run(['suricata', '--build-info'],
                         capture_output=True, text=True)
for line in version.stdout.split('\n')[:10]:
    print(line)

print('\n── Estado del servicio ──')
status = subprocess.run(['systemctl', 'status', 'suricata', '--no-pager', '-l'],
                        capture_output=True, text=True)
print(status.stdout[:800])

In [ ]:
# ── Verificar reglas cargadas ─────────────────────────────────────────────────
import glob

RULES_DIR = '/var/lib/suricata/rules'
rule_files = glob.glob(f'{RULES_DIR}/*.rules')

if not rule_files:
    # fallback: buscar con sudo
    result = subprocess.run(['sudo', 'find', RULES_DIR, '-name', '*.rules'],
                            capture_output=True, text=True)
    rule_files = [l.strip() for l in result.stdout.splitlines() if l.strip()]

total_rules = 0
print('Archivos de reglas cargados:')
for rf in rule_files:
    try:
        result = subprocess.run(['sudo', 'grep', '-c', '^alert', rf],
                                capture_output=True, text=True)
        count = int(result.stdout.strip()) if result.returncode == 0 else 0
    except Exception:
        count = 0
    total_rules += count
    print(f'  {os.path.basename(rf):40s}: {count:6d} reglas')

print(f'\nTotal de reglas activas: {total_rules:,}')

## Sección 4: Primera Operación – Reconocimiento

Vamos a registrar un agente local y ejecutar una operación de reconocimiento simple:
- **T1082**: System Information Discovery
- **T1057**: Process Discovery
- **T1033**: System Owner/User Discovery


In [ ]:
# ── Descargar y ejecutar el agente Sandcat ────────────────────────────────────
# El agente se registra automáticamente en Caldera

import subprocess, time, threading

AGENT_SCRIPT = '/tmp/caldera_agent.sh'

agent_cmd = f'''#!/bin/bash
server="http://localhost:8888"
curl -s --max-time 15 -X POST -H 'file:sandcat.go-linux' \
     -H 'platform:linux' \
     $server/file/download > /tmp/sandcat 2>/dev/null
chmod +x /tmp/sandcat
/tmp/sandcat -server $server -group red -v >/tmp/sandcat.log 2>&1 &
disown $!
echo "Agent started with PID $!"
'''

with open(AGENT_SCRIPT, 'w') as f:
    f.write(agent_cmd)

result = subprocess.run(['bash', AGENT_SCRIPT], capture_output=True, text=True, timeout=30)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[:200])

# Esperar a que el agente se registre
print('Waiting 10s for agent registration...')
time.sleep(10)

In [ ]:
# ── Verificar agentes registrados ─────────────────────────────────────────────
r = requests.get(f'{CALDERA_URL}/api/v2/agents', headers=HEADERS, timeout=10)
agents = r.json() if r.status_code == 200 else []
print(f'Agentes registrados: {len(agents)}')
for agent in agents:
    print(f"  paw={agent.get('paw','?'):12s} host={agent.get('host','?'):20s} "
          f"platform={agent.get('platform','?'):10s} status={agent.get('sleep_min','?')}")

In [ ]:
# ── Crear adversario de reconocimiento ────────────────────────────────────────
# Buscar abilities de discovery/reconnaissance con soporte Linux
recon_abilities = [
    a for a in abilities
    if a.get('tactic') in ('discovery', 'reconnaissance')
    and any(e.get('platform') == 'linux' for e in a.get('executors', []))
][:5]

if not recon_abilities:
    print('No se encontraron discovery abilities para Linux – usando cualquier plataforma')
    recon_abilities = [
        a for a in abilities
        if a.get('tactic') in ('discovery', 'reconnaissance')
    ][:5]

recon_ability_ids = [a['ability_id'] for a in recon_abilities]
print('Abilities seleccionadas para reconocimiento:')
for a in recon_abilities:
    platforms = [e.get('platform') for e in a.get('executors', [])]
    print(f"  [{a.get('technique_id','?')}] {a.get('name','?')} | plataformas: {platforms}")

# Crear el adversario
adv_payload = {
    'name': 'Lab Recon Adversary',
    'description': 'Simple reconnaissance adversary for IDS practice',
    'atomic_ordering': recon_ability_ids
}

r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                  headers=HEADERS, json=adv_payload, timeout=10)
if r.status_code in (200, 201):
    adv_id = r.json().get('id') or r.json().get('adversary_id')
    print(f'\n✅ Adversario creado: id={adv_id}')
else:
    print(f'Error: {r.status_code} – {r.text[:200]}')
    adv_id = None

In [ ]:
# ── Marcar inicio para monitoreo de logs ──────────────────────────────────────
op_start_time = time.time()

# ── Crear y ejecutar operación ────────────────────────────────────────────────
if adv_id and agents:
    op_payload = {
        'name': 'Recon Operation 1',
        'adversary': {'adversary_id': adv_id},
        'group': 'red',
        'planner': {'id': '4f0610cc-35e7-46ba-9b32-fee69b012913'},  # sequential
        'state': 'running'
    }
    r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                      headers=HEADERS, json=op_payload, timeout=10)
    if r.status_code in (200, 201):
        op_id = r.json().get('id')
        print(f'✅ Operación iniciada: id={op_id}')
        print('   Esperando 30s para que la operación ejecute...')
        time.sleep(30)
    else:
        print(f'Error iniciando operación: {r.status_code} – {r.text[:200]}')
        op_id = None
else:
    print('⚠️ No hay agentes o adversario disponible. Ejecuta las celdas anteriores.')
    op_id = None

In [ ]:
# ── Revisar resultados de la operación ───────────────────────────────────────
if op_id:
    r = requests.get(f'{CALDERA_URL}/api/v2/operations/{op_id}',
                     headers=HEADERS, timeout=10)
    op_data = r.json() if r.status_code == 200 else {}
    print(f"Operación: {op_data.get('name','?')}")
    print(f"Estado:    {op_data.get('state','?')}")
    links = op_data.get('chain', [])
    print(f"Pasos ejecutados: {len(links)}")
    for link in links:
        ability = link.get('ability', {})
        status  = link.get('status', '?')
        cmd     = link.get('command', '')[:80]
        print(f"  [{status:3}] {ability.get('name','?'):40s}  cmd: {cmd}")

## Sección 5: Monitoreo en Tiempo Real con Suricata

In [ ]:
import json
from datetime import datetime

def read_eve_log(filepath=EVE_LOG, event_types=None, since_ts=None, max_events=200):
    """Lee eventos del log eve.json con filtros opcionales."""
    events = []
    if not os.path.exists(filepath):
        print(f'⚠️ {filepath} no encontrado')
        return events
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                evt = json.loads(line)
                if event_types and evt.get('event_type') not in event_types:
                    continue
                if since_ts:
                    ts = evt.get('timestamp', '')
                    evt_t = datetime.fromisoformat(ts.replace('Z','').split('+')[0])
                    if evt_t.timestamp() < since_ts:
                        continue
                events.append(evt)
                if len(events) >= max_events:
                    break
            except json.JSONDecodeError:
                continue
    return events

# Alertas durante la operación
alerts = read_eve_log(event_types=['alert'], since_ts=op_start_time)
print(f'Alertas generadas durante la operación: {len(alerts)}')
for a in alerts[:10]:
    sig = a.get('alert', {}).get('signature', 'N/A')
    src = a.get('src_ip', '?')
    dst = a.get('dest_ip', '?')
    print(f"  {a.get('timestamp','?')[:19]}  {sig[:60]}  {src} → {dst}")

## Sección 6: Operación Intermedia – Cadena Multi-Técnica

Ampliamos el adversario con técnicas de múltiples tácticas:
- **T1003**: OS Credential Dumping
- **T1059**: Command and Scripting Interpreter
- **T1083**: File and Directory Discovery
- **T1005**: Data from Local System


In [ ]:
TARGET_TACTICS = ['discovery', 'credential-access', 'collection', 'execution']

# Filtrar por plataforma Linux y evitar IDs duplicados
seen_ids = set()
multi_abilities = []
for a in abilities:
    if a.get('tactic') in TARGET_TACTICS \
       and any(e.get('platform') == 'linux' for e in a.get('executors', [])) \
       and a['ability_id'] not in seen_ids:
        multi_abilities.append(a)
        seen_ids.add(a['ability_id'])
    if len(multi_abilities) >= 8:
        break

print(f'Abilities seleccionadas ({len(multi_abilities)}):')
for a in multi_abilities:
    print(f"  [{a.get('tactic'):25s}] [{a.get('technique_id','?'):10s}] {a.get('name','?')}")

# Crear adversario multi-técnica
adv2_payload = {
    'name': 'Lab Multi-Technique Adversary',
    'description': 'Multi-tactic adversary: discovery + credential-access + collection',
    'atomic_ordering': [a['ability_id'] for a in multi_abilities]
}

r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                  headers=HEADERS, json=adv2_payload, timeout=10)
if r.status_code in (200, 201):
    adv2_id = r.json().get('id') or r.json().get('adversary_id')
    print(f'\n✅ Adversario multi-técnica: id={adv2_id}')
else:
    print(f'Error {r.status_code}: {r.text[:300]}')
    adv2_id = None

In [ ]:
# ── Ejecutar operación multi-técnica ─────────────────────────────────────────
op2_start = time.time()

if adv2_id and agents:
    op2_payload = {
        'name': 'Multi-Technique Operation',
        'adversary': {'adversary_id': adv2_id},
        'group': 'red',
        'planner': {'id': '4f0610cc-35e7-46ba-9b32-fee69b012913'},
        'state': 'running'
    }
    r = requests.post(f'{CALDERA_URL}/api/v2/operations',
                      headers=HEADERS, json=op2_payload, timeout=10)
    op2_id = r.json().get('id') if r.status_code in (200,201) else None
    print(f'✅ Operación iniciada: id={op2_id}')
    time.sleep(45)
else:
    print('⚠️ Agentes o adversario no disponibles')
    op2_id = None

## Sección 7: Análisis de Logs con Python

Cargamos el `eve.json` completo en un DataFrame de pandas para análisis estadístico.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 100})

def load_eve_json(filepath=EVE_LOG, max_lines=50000):
    """Carga eve.json en un DataFrame de pandas."""
    records = []
    if not os.path.exists(filepath):
        print(f'⚠️ {filepath} no encontrado')
        return pd.DataFrame()
    with open(filepath) as f:
        for i, line in enumerate(f):
            if i >= max_lines:
                break
            try:
                records.append(json.loads(line.strip()))
            except:
                continue
    df = pd.json_normalize(records)
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    return df

df = load_eve_json()
print(f'Total de eventos cargados: {len(df):,}')
if not df.empty:
    print(f'Columnas disponibles: {list(df.columns[:20])}')
    print('\nDistribución por tipo de evento:')
    print(df['event_type'].value_counts())

In [ ]:
# ── Análisis de alertas ───────────────────────────────────────────────────────
if not df.empty and 'event_type' in df.columns:
    df_alerts = df[df['event_type'] == 'alert'].copy()
    print(f'Total de alertas: {len(df_alerts):,}')

    if len(df_alerts) > 0 and 'alert.signature' in df_alerts.columns:
        print('\nTop 10 firmas más frecuentes:')
        print(df_alerts['alert.signature'].value_counts().head(10).to_string())

        print('\nTop 10 categorías de alertas:')
        if 'alert.category' in df_alerts.columns:
            print(df_alerts['alert.category'].value_counts().head(10).to_string())
    else:
        print('No hay alertas aún. Ejecuta una operación primero.')
else:
    print('DataFrame vacío. Verifica que Suricata está generando logs.')

In [ ]:
# ── Visualización: Timeline de alertas ───────────────────────────────────────
if not df.empty and 'event_type' in df.columns:
    df_alerts = df[df['event_type'] == 'alert'].copy()

    if len(df_alerts) > 0 and 'timestamp' in df_alerts.columns:
        df_alerts = df_alerts.dropna(subset=['timestamp'])
        df_alerts['minute'] = df_alerts['timestamp'].dt.floor('T')
        timeline = df_alerts.groupby('minute').size().reset_index(name='count')

        fig, ax = plt.subplots()
        ax.plot(timeline['minute'], timeline['count'], marker='o', linewidth=2)
        ax.fill_between(timeline['minute'], timeline['count'], alpha=0.3)
        ax.set_title('Alertas Suricata en el tiempo')
        ax.set_xlabel('Tiempo')
        ax.set_ylabel('Número de alertas')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print('No hay alertas con timestamp para graficar.')
else:
    print('No hay datos disponibles.')

In [ ]:
# ── Visualización: Top firmas y severidades ───────────────────────────────────
if not df.empty:
    df_alerts = df[df['event_type'] == 'alert'].copy()

    if len(df_alerts) > 0:
        fig, axes = plt.subplots(1, 2)

        # Top categorías
        if 'alert.category' in df_alerts.columns:
            top_cat = df_alerts['alert.category'].value_counts().head(8)
            top_cat.plot(kind='barh', ax=axes[0], color='steelblue')
            axes[0].set_title('Top Categorías de Alertas')
            axes[0].set_xlabel('Frecuencia')

        # Severidades
        if 'alert.severity' in df_alerts.columns:
            sev_counts = df_alerts['alert.severity'].value_counts().sort_index()
            colors = {1: 'red', 2: 'orange', 3: 'yellow', 4: 'green'}
            bar_colors = [colors.get(s, 'gray') for s in sev_counts.index]
            sev_counts.plot(kind='bar', ax=axes[1], color=bar_colors)
            axes[1].set_title('Distribución de Severidades')
            axes[1].set_xlabel('Severidad (1=alta, 4=info)')
            axes[1].set_ylabel('Frecuencia')

        plt.tight_layout()
        plt.show()
    else:
        print('Sin alertas para visualizar.')

## Sección 8: Perspectiva Blue Team – Cobertura de Detección

Analizamos qué técnicas MITRE ATT&CK han generado alertas y calculamos
la cobertura de detección del IDS.

In [ ]:
# ── Mapeo operaciones Caldera → alertas Suricata ─────────────────────────────
all_operations = []
r = requests.get(f'{CALDERA_URL}/api/v2/operations', headers=HEADERS, timeout=10)
if r.status_code == 200:
    all_operations = r.json()

print(f'Operaciones ejecutadas: {len(all_operations)}')

for op in all_operations:
    print(f"\n  OperationID: {op.get('id','?')}")
    print(f"  Nombre:      {op.get('name','?')}")
    print(f"  Estado:      {op.get('state','?')}")
    links = op.get('chain', [])
    print(f"  Pasos:       {len(links)}")

    techniques_used = set()
    for link in links:
        tactic = link.get('ability', {}).get('tactic', '')
        tid    = link.get('ability', {}).get('technique_id', '')
        if tid:
            techniques_used.add((tactic, tid))
    if techniques_used:
        print('  Técnicas:')
        for tactic, tid in sorted(techniques_used):
            print(f'    [{tactic:25s}] {tid}')

In [ ]:
# ── Análisis de cobertura ─────────────────────────────────────────────────────
if not df.empty:
    df_alerts = df[df['event_type'] == 'alert'].copy()
    total_alerts = len(df_alerts)

    print('╔══════════════════════════════════════════════════╗')
    print('║           RESUMEN DE COBERTURA IDS               ║')
    print('╠══════════════════════════════════════════════════╣')
    print(f'║  Total eventos capturados:  {len(df):>8,}           ║')
    print(f'║  Total alertas generadas:   {total_alerts:>8,}           ║')
    if 'alert.severity' in df_alerts.columns and total_alerts > 0:
        sev1 = len(df_alerts[df_alerts['alert.severity'] == 1])
        sev2 = len(df_alerts[df_alerts['alert.severity'] == 2])
        print(f'║  Alertas severidad 1 (Alta): {sev1:>7,}           ║')
        print(f'║  Alertas severidad 2 (Med):  {sev2:>7,}           ║')
    print('╚══════════════════════════════════════════════════╝')
else:
    print('No hay datos de alertas disponibles aún.')

## Sección 9: Operación Avanzada – Persistencia y Movimiento Lateral

Creamos un adversario sofisticado que utiliza:
- **T1053**: Scheduled Task/Job (Persistence)
- **T1021**: Remote Services (Lateral Movement)
- **T1074**: Data Staged (Collection)
- **T1041**: Exfiltration Over C2 Channel


In [ ]:
ADV_TACTICS = ['persistence', 'lateral-movement', 'collection', 'exfiltration', 'impact']

# Filtrar por Linux y deduplicar
seen_ids = set()
adv_abilities = []
for a in abilities:
    if a.get('tactic') in ADV_TACTICS \
       and any(e.get('platform') == 'linux' for e in a.get('executors', [])) \
       and a['ability_id'] not in seen_ids:
        adv_abilities.append(a)
        seen_ids.add(a['ability_id'])
    if len(adv_abilities) >= 6:
        break

print(f'Abilities avanzadas ({len(adv_abilities)}):')
for a in adv_abilities:
    print(f"  [{a.get('tactic'):25s}] [{a.get('technique_id','?'):10s}] {a.get('name','?')}")

# Crear adversario avanzado
adv3_payload = {
    'name': 'Lab Advanced Campaign',
    'description': 'Persistence + Lateral Movement + Exfiltration',
    'atomic_ordering': [a['ability_id'] for a in adv_abilities]
}

r = requests.post(f'{CALDERA_URL}/api/v2/adversaries',
                  headers=HEADERS, json=adv3_payload, timeout=10)
if r.status_code in (200, 201):
    adv3_id = r.json().get('id') or r.json().get('adversary_id')
    print(f'\n✅ Adversario avanzado: id={adv3_id}')
else:
    print(f'Error {r.status_code}: {r.text[:300]}')
    adv3_id = None

## Sección 10: Técnicas Avanzadas – Creación de Reglas Personalizadas

Aprenderemos a escribir reglas Suricata personalizadas para detectar
el comportamiento de los agentes Caldera.

In [ ]:
CUSTOM_RULES = '''\
# ── Reglas personalizadas para SURI-CALDERA Lab ──────────────────────────────

# Detectar peticiones HTTP al servidor Caldera (agente intentando conectar)
alert http any any -> $HOME_NET 8888 (msg:"CALDERA Agent Registration"; content:"sandcat"; http.user_agent; classtype:trojan-activity; sid:9000001; rev:1;)

# Detectar descarga del binario Sandcat
alert http $HOME_NET any -> any any (msg:"CALDERA Sandcat Agent Download"; content:"sandcat.go"; http.uri; classtype:trojan-activity; sid:9000002; rev:1;)

# Detectar comandos de reconocimiento comunes
alert http any any -> $HOME_NET 8888 (msg:"CALDERA Recon Command Received"; content:"POST"; http.method; content:"/api/v2/instructions"; http.uri; classtype:policy-violation; sid:9000003; rev:1;)

# Detectar tráfico de beacon (check-in periódico de agentes)
alert http any any -> $HOME_NET 8888 (msg:"CALDERA Agent Beacon"; content:"GET"; http.method; content:"/api/v2/instructions"; http.uri; threshold: type both, track by_src, count 5, seconds 60; classtype:trojan-activity; sid:9000004; rev:1;)
'''

# Las reglas de Suricata se guardan en /var/lib/suricata/rules/
custom_rules_path = '/var/lib/suricata/rules/custom-caldera.rules'

result = subprocess.run(
    ['sudo', 'bash', '-c', f'cat > {custom_rules_path}'],
    input=CUSTOM_RULES, capture_output=True, text=True
)

if result.returncode == 0:
    print(f'✅ Reglas personalizadas guardadas en {custom_rules_path}')
    print('\nContenido:')
    print(CUSTOM_RULES)
else:
    print(f'Error guardando reglas: {result.stderr[:300]}')
    print('\nReglas que se aplicarían:')
    print(CUSTOM_RULES)

In [ ]:
# ── Recargar reglas en Suricata (sin reiniciar el servicio) ───────────────────

# Enviar SIGUSR2 directamente con el PID
pid_result = subprocess.run(['sudo', 'cat', '/var/run/suricata/suricata.pid'],
                            capture_output=True, text=True)
suricata_pid = pid_result.stdout.strip()

if suricata_pid.isdigit():
    reload = subprocess.run(['sudo', 'kill', '-USR2', suricata_pid],
                            capture_output=True, text=True)
    print(f'SIGUSR2 enviado al PID {suricata_pid}: {"OK" if reload.returncode == 0 else reload.stderr}')
else:
    # Fallback: reiniciar el servicio
    reload = subprocess.run(['sudo', 'systemctl', 'reload-or-restart', 'suricata'],
                            capture_output=True, text=True)
    print(f'Reload via systemctl: {"OK" if reload.returncode == 0 else reload.stderr}')

# Verificar configuración válida
check = subprocess.run(
    ['sudo', 'suricata', '-T', '-c', '/etc/suricata/suricata.yaml'],
    capture_output=True, text=True
)
if check.returncode == 0:
    print('✅ Configuración de Suricata válida')
else:
    print('Salida test:', (check.stdout + check.stderr)[-500:])

## Sección 11: Caso de Estudio – Campaña APT Simulada

Simulamos una campaña APT completa con las fases:
1. Reconocimiento inicial
2. Establecimiento de persistencia
3. Movimiento lateral
4. Exfiltración de datos
5. Análisis forense post-incidente

In [ ]:
from datetime import datetime, timezone

# ── Crear adversario APT completo ─────────────────────────────────────────────
# Combinar todas las abilities disponibles en una campaña
apt_tactics = ['reconnaissance', 'discovery', 'credential-access',
               'persistence', 'collection', 'exfiltration']
apt_abilities = []
for tactic in apt_tactics:
    for a in abilities:
        if a.get('tactic') == tactic and len(apt_abilities) < 12:
            apt_abilities.append(a)
            break

print(f'Campaña APT simulada – {len(apt_abilities)} técnicas:')
for i, a in enumerate(apt_abilities, 1):
    tactic = a.get('tactic', '?')
    tid    = a.get('technique_id', '?')
    name   = a.get('name', '?')
    print(f'  Fase {i:2d}: [{tactic:25s}] [{tid:10s}] {name}')

apt_start = datetime.now(timezone.utc)
print(f'\nInicio de campaña: {apt_start.isoformat()}')

In [ ]:
# ── Análisis forense post-campaña ─────────────────────────────────────────────
print('='*60)
print('  ANÁLISIS FORENSE POST-INCIDENTE')
print('='*60)

# Cargar todos los eventos del período
df_fresh = load_eve_json()

if not df_fresh.empty:
    event_types = df_fresh['event_type'].value_counts() if 'event_type' in df_fresh.columns else pd.Series()
    print('\nEventos capturados por tipo:')
    print(event_types.to_string())

    if 'event_type' in df_fresh.columns:
        df_alerts_fresh = df_fresh[df_fresh['event_type'] == 'alert'].copy()

        # IPs de origen más activas
        if 'src_ip' in df_alerts_fresh.columns and len(df_alerts_fresh) > 0:
            print('\nTop IPs de origen (alertas):')
            print(df_alerts_fresh['src_ip'].value_counts().head(5).to_string())

        # Protocolos involucrados
        if 'proto' in df_alerts_fresh.columns and len(df_alerts_fresh) > 0:
            print('\nProtocolos en alertas:')
            print(df_alerts_fresh['proto'].value_counts().to_string())
else:
    print('No hay eventos disponibles aún.')

## Sección 12: Exportación y Reporte Final

In [ ]:
import json
from datetime import datetime, timezone

# ── Generar reporte en JSON ───────────────────────────────────────────────────
report = {
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'lab': 'SURI-CALDERA IDS Practice',
    'summary': {}
}

# Resumen de Caldera
ops_r = requests.get(f'{CALDERA_URL}/api/v2/operations', headers=HEADERS, timeout=10)
ops   = ops_r.json() if ops_r.status_code == 200 else []
report['summary']['caldera'] = {
    'operations': len(ops),
    'operation_names': [o.get('name') for o in ops]
}

# Resumen de Suricata
df_rep = load_eve_json()
if not df_rep.empty and 'event_type' in df_rep.columns:
    df_al = df_rep[df_rep['event_type'] == 'alert']
    report['summary']['suricata'] = {
        'total_events': len(df_rep),
        'total_alerts': len(df_al),
        'top_signatures': df_al['alert.signature'].value_counts().head(5).to_dict()
            if 'alert.signature' in df_al.columns else {}
    }

report_path = '/home/vagrant/notebooks/lab_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print('✅ Reporte generado:')
print(json.dumps(report, indent=2, default=str)[:800])
print(f'\n📄 Guardado en: {report_path}')

In [ ]:
# ── Resumen final del laboratorio ─────────────────────────────────────────────
print('╔══════════════════════════════════════════════════════════════╗')
print('║              SURI-CALDERA IDS LAB – COMPLETADO!             ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  ✅ Caldera instalado y operativo                           ║')
print('║  ✅ Suricata detectando tráfico en tiempo real              ║')
print('║  ✅ Operaciones de adversarios ejecutadas                   ║')
print('║  ✅ Logs analizados con Python/pandas                       ║')
print('║  ✅ Reglas personalizadas creadas                           ║')
print('║  ✅ Reporte generado                                        ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  Técnicas MITRE ATT&CK practicadas:                        ║')
print('║    Discovery | Credential Access | Persistence             ║')
print('║    Lateral Movement | Collection | Exfiltration            ║')
print('╚══════════════════════════════════════════════════════════════╝')